# Raw CIC-IDS2017 dataset inspection

This notebook audits the original CSV files in `data/raw/`. It reports row counts, label distributions, missing and infinite values, invalid numeric rows, duplicate feature columns, zero-variance columns, and the two infiltration-file flag columns.

The audit reads the source CSVs but does not clean, save, or overwrite any dataset. The prepared identified-flow dataset is audited separately in `02_identified_flow_audit.ipynb`.

Run the single audit code cell below from the project root or the `notebooks` directory.

In [27]:
from pathlib import Path
import numpy as np
import pandas as pd

# Find the project root whether Jupyter starts in the root or notebooks/.
cwd = Path.cwd()
if (cwd / "data" / "raw").is_dir():
    PROJECT_DIR = cwd
elif (cwd.parent / "data" / "raw").is_dir():
    PROJECT_DIR = cwd.parent
else:
    raise FileNotFoundError(
        "Cannot find data/raw. Start Jupyter from the project root "
        "or the notebooks directory."
    )

DATA_DIR = PROJECT_DIR / "data" / "raw"
files = sorted(DATA_DIR.glob("*.csv"))

if not files:
    raise FileNotFoundError(f"No CSV files found in {DATA_DIR}")

print("Raw data directory:", DATA_DIR)
print("CSV files found:", len(files))

total_physical_rows = 0
total_empty_rows = 0
total_labelled_rows = 0
total_missing_cells = 0
total_infinite_values = 0
total_invalid_numeric_rows = 0

all_label_counts = {}
constant_columns_by_file = {}
invalid_labels_by_file = {}

for file_path in files:
    # latin1 handles the non-UTF-8 bytes present in some original files.
    df = pd.read_csv(file_path, encoding="latin1", low_memory=False)
    df.columns = df.columns.str.strip()

    if "Label" not in df.columns:
        raise KeyError(f"'Label' column not found in {file_path.name}")

    physical_rows = len(df)
    empty_rows = int(df.isna().all(axis=1).sum())
    total_physical_rows += physical_rows
    total_empty_rows += empty_rows

    # Exclude fully empty trailing rows from the data-quality analysis.
    df = df.dropna(how="all").copy()
    df["Label"] = (
        df["Label"].astype("string").str.strip()
        .str.replace("\ufffd", "-", regex=False)
        .str.replace("\x96", "-", regex=False)
    )

    labelled = df["Label"].notna() & df["Label"].ne("")
    total_labelled_rows += int(labelled.sum())

    counts = df.loc[labelled, "Label"].value_counts()
    for label, count in counts.items():
        all_label_counts[label] = all_label_counts.get(label, 0) + int(count)

    total_missing_cells += int(df.isna().sum().sum())

    numeric = df.select_dtypes(include="number")
    infinite_count = int(np.isinf(numeric.to_numpy()).sum())
    total_infinite_values += infinite_count

    numeric_checked = numeric.replace([np.inf, -np.inf], np.nan)
    invalid_mask = numeric_checked.isna().any(axis=1)
    invalid_count = int(invalid_mask.sum())
    total_invalid_numeric_rows += invalid_count

    if invalid_count:
        invalid_labels_by_file[file_path.name] = (
            df.loc[invalid_mask, "Label"].value_counts(dropna=False).to_dict()
        )

    constant_columns = numeric.columns[
        numeric.nunique(dropna=False) <= 1
    ].tolist()
    if constant_columns:
        constant_columns_by_file[file_path.name] = constant_columns

    duplicate_feature = None
    if {
        "Fwd Header Length.1",
        "Fwd Header Length",
    }.issubset(df.columns):
        duplicate_feature = df["Fwd Header Length.1"].equals(
            df["Fwd Header Length"]
        )

    print(f"\n--- {file_path.name} ---")
    print("Physical rows:", physical_rows)
    print("Fully empty rows:", empty_rows)
    print("Labelled, non-empty rows:", int(labelled.sum()))
    print("Missing cells:", int(df.isna().sum().sum()))
    print("Infinite numeric values:", infinite_count)
    print("Rows with invalid numeric features:", invalid_count)
    print("Label distribution:")
    print(counts.to_string() if not counts.empty else "(no labelled rows)")
    if duplicate_feature is not None:
        print("Fwd Header Length.1 exactly duplicates Fwd Header Length:",
              duplicate_feature)
    if constant_columns:
        print("Zero-variance numeric columns:", constant_columns)

print("\n========== COMBINED AUDIT ==========")
print("Files:", len(files))
print("Physical rows across CSVs:", total_physical_rows)
print("Fully empty rows:", total_empty_rows)
print("Labelled, non-empty rows:", total_labelled_rows)
print("Missing cells in non-empty rows:", total_missing_cells)
print("Infinite numeric values:", total_infinite_values)
print("Rows with missing/infinite numeric features:",
      total_invalid_numeric_rows)

print("\nCombined label counts:")
print(pd.Series(all_label_counts).sort_values(ascending=False).to_string())

print("\nInvalid numeric rows by label and file:")
if invalid_labels_by_file:
    for filename, counts in invalid_labels_by_file.items():
        print(f"\n{filename}")
        print(pd.Series(counts).to_string())
else:
    print("None found.")

print("\nZero-variance numeric columns by file:")
for filename, columns in constant_columns_by_file.items():
    print(f"{filename}: {columns}")

# Investigate the two flags in the Thursday infiltration file.
infiltration_files = [
    p for p in files if "Thursday-WorkingHours-Afternoon-Infilteration" in p.name
]

if infiltration_files:
    infil = pd.read_csv(
        infiltration_files[0], encoding="latin1", low_memory=False
    )
    infil.columns = infil.columns.str.strip()
    infil = infil.dropna(how="all").copy()
    infil["Label"] = (
        infil["Label"].astype("string").str.strip()
        .str.replace("\ufffd", "-", regex=False)
        .str.replace("\x96", "-", regex=False)
    )

    print("\nInfiltration flag cross-tabs:")
    for column in ["Fwd URG Flags", "CWE Flag Count"]:
        if column in infil.columns:
            print(f"\n{column}")
            print(pd.crosstab(infil[column], infil["Label"]))
else:
    print("\nInfiltration file not found; flag cross-tabs skipped.")


Raw data directory: F:\Project\IDS-ML-Thesis\data\raw
CSV files found: 8

--- Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv ---
Physical rows: 225745
Fully empty rows: 0
Labelled, non-empty rows: 225745
Missing cells: 4
Infinite numeric values: 64
Rows with invalid numeric features: 34
Label distribution:
Label
DDoS      128027
BENIGN     97718
Fwd Header Length.1 exactly duplicates Fwd Header Length: True
Zero-variance numeric columns: ['Bwd PSH Flags', 'Fwd URG Flags', 'Bwd URG Flags', 'CWE Flag Count', 'Fwd Avg Bytes/Bulk', 'Fwd Avg Packets/Bulk', 'Fwd Avg Bulk Rate', 'Bwd Avg Bytes/Bulk', 'Bwd Avg Packets/Bulk', 'Bwd Avg Bulk Rate']

--- Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv ---
Physical rows: 286467
Fully empty rows: 0
Labelled, non-empty rows: 286467
Missing cells: 15
Infinite numeric values: 727
Rows with invalid numeric features: 371
Label distribution:
Label
PortScan    158930
BENIGN      127537
Fwd Header Length.1 exactly duplicates Fwd Header Length: True
Z

## Legacy preprocessing reference

The old preprocessing code has been removed from this active notebook to prevent accidental overwriting of `data/processed/clean_flows.parquet`.

The pre-cleanup notebook is preserved at `notebooks/backups/01_dataset_inspection_before_cleanup.ipynb`. Do not run its old preprocessing cells: they write to the legacy processed-data path.

The current identified-flow preparation script is `src/prepare_identified_flows.py`.

## Historical experiment: random train/test split

A random stratified split was explored during initial inspection. It is not the final evaluation protocol because related network flows can appear in both training and test sets.

Evaluation protocols will be defined explicitly in the experiment notebook. Do not use this historical note as a final model evaluation procedure.